In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import cv2
import os
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# --- CONFIGURATION ---
EXPERIMENT_MODE = "R2"  # Or "R0"
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./STANDUP_Ready_Phase2"
PRETRAINED_PATH = "best_inaoe_encoder_v2.pth"

# --- 1. DATASET LOADER ---
class SiameseFootDataset(Dataset):
    def __init__(self, root_dir, mode='train', experiment_mode='R2'):
        self.root_dir = root_dir
        self.mode = mode
        self.experiment_mode = experiment_mode
        self.pairs = self._make_pairs()
        
        if self.mode == 'train':
            self.transform = A.Compose([
                A.Rotate(limit=10, p=0.5), 
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=0, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
                A.Resize(256, 256),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])
        else:
            self.transform = A.Compose([
                A.Resize(256, 256),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])
        
    def _make_pairs(self):
        pairs = []
        for root, dirs, files in os.walk(self.root_dir):
            for f in files:
                if "_L.png" in f:
                    l_path = os.path.join(root, f)
                    r_path = os.path.join(root, f.replace("_L.png", "_R.png"))
                    
                    if os.path.exists(r_path):
                        folder_name = os.path.basename(root).lower()
                        label = None
                        if "healthy" in folder_name: label = 0.0
                        elif self.experiment_mode == "R2" and "diabetic_r2" in folder_name: label = 1.0
                        elif self.experiment_mode == "R0" and "diabetic_r0" in folder_name: label = 1.0
                            
                        if label is not None: pairs.append((l_path, r_path, label))
        return pairs

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        l_path, r_path, label = self.pairs[idx]
        img_l = cv2.imread(l_path, cv2.IMREAD_GRAYSCALE)
        img_r = cv2.imread(r_path, cv2.IMREAD_GRAYSCALE)
        
        if img_l is None or img_r is None:
            return torch.zeros(1, 256, 256), torch.zeros(1, 256, 256), torch.tensor(label).float()

        return self.transform(image=img_l)["image"], self.transform(image=img_r)["image"], torch.tensor(label).float()

# --- 2. MODELS ---
class SpatialTransformer(nn.Module):
    def __init__(self):
        super(SpatialTransformer, self).__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=7), nn.MaxPool2d(2, stride=2), nn.ReLU(True),
            nn.Conv2d(8, 10, kernel_size=5), nn.MaxPool2d(2, stride=2), nn.ReLU(True)
        )
        self.fc_loc = nn.Sequential(nn.Linear(10 * 60 * 60, 32), nn.ReLU(True), nn.Linear(32, 3 * 2))
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 60 * 60)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        return F.grid_sample(x, grid, align_corners=True)

class SiameseNetwork(nn.Module):
    def __init__(self, pretrained_path=None, use_stn=True):
        super(SiameseNetwork, self).__init__()
        self.use_stn = use_stn
        
        if self.use_stn:
            self.stn = SpatialTransformer()
            
        # CRITICAL FIX 1: Explicitly set encoder_weights=None so the baseline is truly random!
        self.encoder = smp.Unet(encoder_name="resnet18", encoder_weights=None, in_channels=1, classes=1).encoder
        
        if pretrained_path and os.path.exists(pretrained_path):
            self.encoder.load_state_dict(torch.load(pretrained_path))
            
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    def forward(self, l, r):
        r_flipped = torch.flip(r, dims=[3])
        
        # CRITICAL FIX 2: If using STN with Pretrained weights, we detach the STN gradients 
        # from the encoder initially, or just let it align normally. 
        r_aligned = self.stn(r_flipped) if self.use_stn else r_flipped
        
        f_l = self.encoder(l)[-1].mean(dim=(2,3)) 
        f_r = self.encoder(r_aligned)[-1].mean(dim=(2,3))
        return self.classifier(torch.abs(f_l - f_r))

# --- 3. METRICS HELPER ---
def calculate_metrics(loader, model, device):
    model.eval()
    all_labels, all_scores = [], []
    with torch.no_grad():
        for l, r, label in loader:
            score = model(l.to(device), r.to(device)).cpu().numpy().flatten()
            all_scores.extend(score)
            all_labels.extend(label.numpy().flatten())
            
    all_scores, all_labels = np.array(all_scores), np.array(all_labels)
    preds = (all_scores > 0.5).astype(int)
    
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, zero_division=0)
    try: auc = roc_auc_score(all_labels, all_scores)
    except: auc = 0.5 
    return acc, f1, auc

# --- 4. MAIN ABLATION RUNNER ---
if __name__ == "__main__":
    print(f"🚀 STARTING ABLATION STUDY ({EXPERIMENT_MODE})")
    
    # Dataset prep
    full_dataset = SiameseFootDataset(DATA_DIR, mode='train', experiment_mode=EXPERIMENT_MODE)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    
    # Use fixed generator for reproducibility across ablations
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)
    
    # Disable augmentations on val split manually
    val_dataset.dataset.mode = 'val'
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Define the 4 ablation conditions
    experiments = [
        {"name": "Baseline (No STN, Random Init)", "use_stn": False, "pretrained_path": None},
        {"name": "STN Only (No Pretraining)",      "use_stn": True,  "pretrained_path": None},
        {"name": "Pretrained Only (No STN)",       "use_stn": False, "pretrained_path": PRETRAINED_PATH},
        {"name": "Proposed (STN + Pretrained)",    "use_stn": True,  "pretrained_path": PRETRAINED_PATH},
    ]
    
    results = []

    for exp in experiments:
        print(f"\n=========================================")
        print(f"🔬 Running: {exp['name']}")
        print(f"=========================================")
        
        model = SiameseNetwork(pretrained_path=exp['pretrained_path'], use_stn=exp['use_stn']).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCELoss()
        
        best_f1, best_acc, best_auc = 0.0, 0.0, 0.0
        
        for epoch in range(EPOCHS):
            model.train()
            for l, r, label in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(l.to(DEVICE), r.to(DEVICE)), label.to(DEVICE).unsqueeze(1))
                loss.backward()
                optimizer.step()
                
            # Evaluate
            acc, f1, auc = calculate_metrics(val_loader, model, DEVICE)
            
            # Save best metrics for this experiment
            if f1 >= best_f1:
                best_f1, best_acc, best_auc = f1, acc, auc
                
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Val F1: {f1:.4f}")
                
        results.append({
            "Experiment": exp['name'],
            "F1": best_f1,
            "Accuracy": best_acc,
            "AUROC": best_auc
        })

    # --- PRINT FINAL TABLE ---
    print("\n\n🏆 ABLATION STUDY RESULTS SUMMARY 🏆")
    print("-" * 75)
    print(f"{'Model Configuration':<35} | {'F1 Score':<10} | {'Accuracy':<10} | {'AUROC':<10}")
    print("-" * 75)
    for r in results:
        print(f"{r['Experiment']:<35} | {r['F1']:.4f}     | {r['Accuracy']:.4f}     | {r['AUROC']:.4f}")
    print("-" * 75)